<a href="https://colab.research.google.com/github/nguyentuanphong0510/Machine-Learning/blob/main/BTOL_Buoi02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Bài 1
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Tải dữ liệu Iris
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Tìm K tối ưu (có và không có StandardScaler)
k_values = range(1, 16)
acc_raw = []
acc_scaled = []

for k in k_values:
    # Không chuẩn hóa
    m_raw = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    m_raw.fit(X_train, y_train)
    acc_raw.append(accuracy_score(y_test, m_raw.predict(X_test)))

    # Có chuẩn hóa StandardScaler
    m_sc = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, metric='euclidean'))
    m_sc.fit(X_train, y_train)
    acc_scaled.append(accuracy_score(y_test, m_sc.predict(X_test)))

df_k = pd.DataFrame({
    "K": list(k_values),
    "Acc_Khong_Chuan_Hoa": np.round(acc_raw, 4),
    "Acc_Co_StandardScaler": np.round(acc_scaled, 4)
})
print("=== So sánh Accuracy theo K ===")
print(df_k.to_string(index=False))

best_k = k_values[int(np.argmax(acc_scaled))]
print(f"\n-> K tốt nhất (với StandardScaler): K = {best_k}, Accuracy = {max(acc_scaled):.4f}")

# 3. Thực nghiệm các độ đo khoảng cách (Euclidean, Manhattan, Minkowski p=3) với best_k
metrics_list = [
    ("euclidean", {}),
    ("manhattan", {}),
    ("minkowski", {"p": 3})
]

print("\n=== So sánh các độ đo khoảng cách ===")
for metric_name, kw in metrics_list:
    clf = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=best_k, metric=metric_name, **kw)
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f"Metric = {metric_name} {kw} -> Accuracy = {accuracy_score(y_test, y_pred):.4f}")

=== So sánh Accuracy theo K ===
 K  Acc_Khong_Chuan_Hoa  Acc_Co_StandardScaler
 1               0.9667                 0.9667
 2               0.9333                 0.9333
 3               1.0000                 0.9333
 4               1.0000                 0.9333
 5               1.0000                 0.9333
 6               0.9667                 0.9333
 7               0.9667                 0.9667
 8               0.9667                 0.9333
 9               1.0000                 0.9667
10               1.0000                 0.9667
11               0.9667                 0.9667
12               0.9667                 0.9667
13               0.9667                 0.9667
14               0.9667                 0.9667
15               0.9667                 0.9667

-> K tốt nhất (với StandardScaler): K = 1, Accuracy = 0.9667

=== So sánh các độ đo khoảng cách ===
Metric = euclidean {} -> Accuracy = 0.9667
Metric = manhattan {} -> Accuracy = 0.9667
Metric = minkowski {'p': 3} -

In [ ]:
#Bài 2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Tải dữ liệu Boston Housing (hoặc upload file CSV từ Kaggle và dùng pd.read_csv("BostonHousing.csv"))
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url).dropna()

target_col = 'medv' if 'medv' in df.columns else 'MEDV'
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. Chia tập huấn luyện (80%) và tập kiểm tra (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3 & 4 & 5. Chuẩn hóa, chạy KNN Regressor với k = 3, 5, 7, 9 và tính MSE, R2
k_list = [3, 5, 7, 9]
results = []

for k in k_list:
    knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    knn_reg.fit(X_train, y_train)
    y_pred = knn_reg.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({"k": k, "MSE": round(mse, 4), "R2_score": round(r2, 4)})

print("=== Kết quả KNN Regressor trên Boston Housing ===")
print(pd.DataFrame(results).to_string(index=False))

=== Kết quả KNN Regressor trên Boston Housing ===
 k     MSE  R2_score
 3 19.4043    0.7354
 5 20.6055    0.7190
 7 23.4155    0.6807
 9 24.0094    0.6726


In [ ]:
#bài 3
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X, y = iris.data, iris.target

# Thực nghiệm nhiều tỷ lệ chia train/test và trạng thái ngẫu nhiên
splits = [
    ("90/10 (seed=42)", 0.1, 42, None),
    ("80/20 (seed=42)", 0.2, 42, None),
    ("70/30 (seed=42)", 0.3, 42, None),
    ("80/20 stratify (seed=42)", 0.2, 42, y),
    ("80/20 stratify (seed=100)", 0.2, 100, y),
]

rows = []
for name, test_sz, seed, strat in splits:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_sz, random_state=seed, stratify=strat
    )
    svm_clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, gamma='scale'))
    svm_clf.fit(X_tr, y_tr)
    y_pr = svm_clf.predict(X_te)
    acc = accuracy_score(y_te, y_pr)
    rows.append({"Cách chia": name, "Số mẫu test": len(y_te), "Accuracy": round(acc, 4)})

print("=== So sánh SVM (RBF) trên Iris theo các cách chia dữ liệu ===")
print(pd.DataFrame(rows).to_string(index=False))

=== So sánh SVM (RBF) trên Iris theo các cách chia dữ liệu ===
                Cách chia  Số mẫu test  Accuracy
          90/10 (seed=42)           15    1.0000
          80/20 (seed=42)           30    1.0000
          70/30 (seed=42)           45    1.0000
 80/20 stratify (seed=42)           30    0.9667
80/20 stratify (seed=100)           30    0.9000


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url).dropna()

target_col = 'medv' if 'medv' in df.columns else 'MEDV'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

svr_rows = []
for kernel_type in ['linear', 'rbf']:
    svr_model = make_pipeline(StandardScaler(), SVR(kernel=kernel_type, C=10.0))
    svr_model.fit(X_train, y_train)
    y_pred = svr_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    svr_rows.append({
        "Kernel": kernel_type,
        "MSE": round(mse, 4),
        "R2_score": round(r2, 4)
    })

print("=== Kết quả SVR trên Boston Housing ===")
print(pd.DataFrame(svr_rows).to_string(index=False))

=== Kết quả SVR trên Boston Housing ===
Kernel     MSE  R2_score
linear 28.7853    0.6075
   rbf 12.7382    0.8263
